### Célula 1 — Importar e verificar o pipeline

In [1]:
import sys
sys.path.append('..')

from src.pipeline import (run_preprocessing, run_feature_engineering,
                           run_segmentation, run_prediction,
                           run_reports, run_pipeline, PATHS)

print("Pipeline importado com sucesso!")
print(f"\nCaminhos configurados:")
for chave, valor in PATHS.items():
    print(f"  {chave}: {valor}")

Pipeline importado com sucesso!

Caminhos configurados:
  raw: ../data/raw/online_retail_II.xlsx
  processed: ../data/processed/online_retail_clean.csv
  features: ../data/features/rfm_segmentado.csv
  predictions: ../data/features/rfm_com_predicoes.csv
  kmeans: ../models/kmeans_rfm.pkl
  scaler_seg: ../models/scaler_rfm.pkl
  rf_model: ../models/random_forest_churn.pkl
  scaler_pred: ../models/scaler_prediction.pkl


### Célula 2 — Testar etapas isoladas antes do pipeline completo

In [2]:
import pandas as pd

# Carregar dados já processados — evita rodar o Excel novamente
df = pd.read_csv('../data/processed/online_retail_clean.csv',
                 parse_dates=['InvoiceDate'])

print(f"Dados limpos carregados: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Período: {df['InvoiceDate'].min().date()} até {df['InvoiceDate'].max().date()}")

Dados limpos carregados: 770,715 linhas x 9 colunas
Período: 2009-12-01 até 2011-12-09


### Célula 3 — Testar Feature Engineering

In [3]:
from src.feature_engineering import calculate_rfm, apply_log_transform

rfm = calculate_rfm(df)
rfm_log = apply_log_transform(rfm)

print(f"RFM calculado: {len(rfm):,} clientes")
print(f"\nColunas RFM: {list(rfm.columns)}")
print(f"Colunas RFM log: {list(rfm_log.columns)}")
print(f"\nAmostra:")
rfm.head()

RFM calculado: 5,816 clientes

Colunas RFM: ['Customer ID', 'Recency', 'Frequency', 'Monetary']
Colunas RFM log: ['Customer ID', 'Recency', 'Frequency', 'Monetary', 'Recency_log', 'Frequency_log', 'Monetary_log']

Amostra:


,Customer ID,Recency,Frequency,Monetary
0,12346,529,3,170.36
1,12347,2,8,4671.93
2,12348,75,5,1658.40
3,12349,19,3,3678.69
4,12350,310,1,294.40


### Célula 4 — Testar Segmentação

In [4]:
from src.segmentation import scale_features, train_kmeans, assign_segments

rfm_scaled, scaler = scale_features(rfm_log)
kmeans = train_kmeans(rfm_scaled, n_clusters=4)
rfm = assign_segments(rfm, kmeans, rfm_scaled)

print(f"Segmentação concluída!")
print(f"\nDistribuição dos segmentos:")
print(rfm['Segmento'].value_counts())
print(f"\nAmostra:")
rfm[['Customer ID', 'Recency', 'Frequency', 'Monetary', 'Segmento']].head()

Segmentação concluída!

Distribuição dos segmentos:
Segmento
Perdido      1968
Em Risco     1432
Promissor    1237
VIP          1179
Name: count, dtype: int64

Amostra:


,Customer ID,Recency,Frequency,Monetary,Segmento
0,12346,529,3,170.36,Perdido
1,12347,2,8,4671.93,VIP
2,12348,75,5,1658.40,Em Risco
3,12349,19,3,3678.69,Promissor
4,12350,310,1,294.40,Perdido


### Célula 5 — Testar Predição

In [5]:
from src.prediction import (create_churn_label, prepare_features,
                             scale_features as scale_pred,
                             train_model, evaluate_model, predict_churn)
from sklearn.model_selection import train_test_split

rfm = create_churn_label(rfm)
X, y = prepare_features(rfm)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_scaled, X_test_scaled, scaler_pred = scale_pred(X_train, X_test)
rf = train_model(X_train_scaled, y_train)
metrics = evaluate_model(rf, X_test_scaled, y_test)

Resultado do modelo:
              precision    recall  f1-score   support

       Ativo       0.98      0.98      0.98       484
       Churn       0.99      0.99      0.99       680

    accuracy                           0.98      1164
   macro avg       0.98      0.98      0.98      1164
weighted avg       0.98      0.98      0.98      1164

ROC-AUC: 0.9993


### Célula 6 — Testar Relatórios

In [6]:
from src.prediction import predict_churn
from src.evaluation import segment_report, churn_report

rfm = predict_churn(rfm, rf, scaler_pred)

print("RELATÓRIO DE SEGMENTOS:")
print('='*55)
print(segment_report(rfm).to_string())

print("\nRELATÓRIO DE CHURN:")
print('='*55)
print(churn_report(rfm).to_string())

RELATÓRIO DE SEGMENTOS:
           Clientes  Recency_media  Frequency_media  Monetary_media  Receita_total  Pct_Receita
Segmento                                                                                       
Em Risco       1432         226.64             5.08         1813.29     2596629.46        18.12
Perdido        1968         393.50             1.38          310.36      610793.45         4.26
Promissor      1237          28.55             3.00          806.90      998139.25         6.96
VIP            1179          27.12            18.86         8589.31    10126800.92        70.66

RELATÓRIO DE CHURN:
           Clientes  Churn_medio  Alto_risco  Pct_Alto_Risco
Segmento                                                    
Em Risco       1432         0.98        1415           98.81
Perdido        1968         0.99        1963           99.75
Promissor      1237         0.02           3            0.24
VIP            1179         0.02           0            0.00


## O que os relatórios confirmam
Segmentos — consistente com os notebooks

- VIP com 70,66% da receita — números idênticos
- Pipeline reproduzindo resultados com fidelidade total

Churn — os números são contundentes
```
Em Risco  → 98,81% de alto risco  — 1.415 de 1.432 clientes
Perdido   → 99,75% de alto risco  — 1.963 de 1.968 clientes
Promissor → 0,24%  de alto risco  — apenas 3 clientes
VIP       → 0,00%  de alto risco  — zero clientes em risco
```
A separação é quase perfeita — o modelo tem certeza absoluta sobre cada segmento.

### Célula 7 — Teste do pipeline completo de ponta a ponta

In [1]:
import sys
sys.path.append('..')

from src.pipeline import run_pipeline

# Ajustar caminhos para rodar a partir da pasta notebooks/
PATHS_TEST = {
    'raw':         '../data/raw/online_retail_II.xlsx',
    'processed':   '../data/processed/online_retail_clean.csv',
    'features':    '../data/features/rfm_segmentado.csv',
    'predictions': '../data/features/rfm_com_predicoes.csv',
    'kmeans':      '../models/kmeans_rfm.pkl',
    'scaler_seg':  '../models/scaler_rfm.pkl',
    'rf_model':    '../models/random_forest_churn.pkl',
    'scaler_pred': '../models/scaler_prediction.pkl',
}

rfm_final = run_pipeline(paths=PATHS_TEST)

print(f"\nDataset final: {rfm_final.shape[0]:,} clientes x {rfm_final.shape[1]} colunas")
print(f"Colunas: {list(rfm_final.columns)}")


CUSTOMER SEGMENTATION PREDICTION PLATFORM
Iniciando pipeline completo...

ETAPA 1 — PREPROCESSING
Dataset limpo salvo em ../data/processed/online_retail_clean.csv — 770,715 linhas x 9 colunas

ETAPA 2 — FEATURE ENGINEERING (RFM)
RFM salvo em ../data/features/rfm_segmentado.csv — 5,816 clientes
RFM calculado para 5,816 clientes

ETAPA 3 — SEGMENTAÇÃO (K-MEANS)
Modelos salvos:
  → ../models/kmeans_rfm.pkl
  → ../models/scaler_rfm.pkl

Distribuição dos segmentos:
Segmento
Perdido      1968
Em Risco     1432
Promissor    1237
VIP          1179
Name: count, dtype: int64

ETAPA 4 — PREDIÇÃO DE CHURN
Resultado do modelo:
              precision    recall  f1-score   support

       Ativo       0.98      0.98      0.98       484
       Churn       0.99      0.99      0.99       680

    accuracy                           0.98      1164
   macro avg       0.98      0.98      0.98      1164
weighted avg       0.98      0.98      0.98      1164

ROC-AUC: 0.9993
Modelos salvos:
  → ../models/rand